# 03 - Feature Engineering

This notebook builds modeling-ready features from Rossmann raw data.
Goal: create reproducible train/test feature tables for time-series and ML workflows.


## Feature Strategy
1. Merge event rows with store metadata.
2. Add calendar signals (month/week/quarter/cyclical encodings).
3. Create competition and promo timing features.
4. Add lag/rolling sales features by store (train only, leakage-safe via shift).
5. Export cleaned feature tables to `data/processed/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 140)


In [ ]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../reports/figures")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW_DIR / "train.csv", parse_dates=["Date"])
test = pd.read_csv(RAW_DIR / "test.csv", parse_dates=["Date"])
store = pd.read_csv(RAW_DIR / "store.csv")

print("train:", train.shape)
print("test :", test.shape)
print("store:", store.shape)


## Utility Functions
These helpers keep feature definitions centralized and consistent across train/test.


In [ ]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Year"] = out["Date"].dt.year
    out["Month"] = out["Date"].dt.month
    out["Day"] = out["Date"].dt.day
    out["DayOfYear"] = out["Date"].dt.dayofyear
    out["WeekOfYear"] = out["Date"].dt.isocalendar().week.astype(int)
    out["Quarter"] = out["Date"].dt.quarter
    out["IsMonthStart"] = out["Date"].dt.is_month_start.astype(int)
    out["IsMonthEnd"] = out["Date"].dt.is_month_end.astype(int)

    # Cyclical encodings reduce discontinuity for month/week boundaries.
    out["Month_sin"] = np.sin(2 * np.pi * out["Month"] / 12)
    out["Month_cos"] = np.cos(2 * np.pi * out["Month"] / 12)
    out["Week_sin"] = np.sin(2 * np.pi * out["WeekOfYear"] / 52)
    out["Week_cos"] = np.cos(2 * np.pi * out["WeekOfYear"] / 52)
    return out


def add_competition_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Fill sparse competition timing metadata.
    out["CompetitionDistance"] = out["CompetitionDistance"].fillna(out["CompetitionDistance"].median())
    out["CompetitionOpenSinceMonth"] = out["CompetitionOpenSinceMonth"].fillna(1)
    out["CompetitionOpenSinceYear"] = out["CompetitionOpenSinceYear"].fillna(out["Date"].dt.year.min())

    comp_start = pd.to_datetime(
        dict(
            year=out["CompetitionOpenSinceYear"].astype(int),
            month=out["CompetitionOpenSinceMonth"].astype(int),
            day=1,
        ),
        errors="coerce",
    )

    out["CompetitionOpenMonths"] = (
        (out["Date"].dt.year - comp_start.dt.year) * 12
        + (out["Date"].dt.month - comp_start.dt.month)
    )
    out["CompetitionOpenMonths"] = out["CompetitionOpenMonths"].clip(lower=0).fillna(0)

    return out


def add_promo2_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["Promo2"] = out["Promo2"].fillna(0)
    out["Promo2SinceWeek"] = out["Promo2SinceWeek"].fillna(1)
    out["Promo2SinceYear"] = out["Promo2SinceYear"].fillna(out["Date"].dt.year.min())
    out["PromoInterval"] = out["PromoInterval"].fillna("")

    month_map = {
        1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
        7: "Jul", 8: "Aug", 9: "Sept", 10: "Oct", 11: "Nov", 12: "Dec"
    }
    out["MonthName"] = out["Date"].dt.month.map(month_map)

    # Promo2 active if store is on Promo2 and current month appears in PromoInterval.
    out["IsPromo2Month"] = out.apply(
        lambda r: int((r["Promo2"] == 1) and isinstance(r["PromoInterval"], str) and (r["MonthName"] in r["PromoInterval"])),
        axis=1,
    )

    return out.drop(columns=["MonthName"])


def fill_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in ["StoreType", "Assortment", "StateHoliday"]:
        if c in out.columns:
            out[c] = out[c].fillna("Unknown")
    if "Open" in out.columns:
        out["Open"] = out["Open"].fillna(1)
    return out


## Build Base Feature Tables


In [ ]:
train_m = train.merge(store, on="Store", how="left")
test_m = test.merge(store, on="Store", how="left")

for fn in [add_calendar_features, add_competition_features, add_promo2_features, fill_categoricals]:
    train_m = fn(train_m)
    test_m = fn(test_m)

print("train_m:", train_m.shape)
print("test_m :", test_m.shape)


## Lag and Rolling Features (Train Only)

Important leakage rule:
- We sort by (`Store`, `Date`) and use `shift(1)` before any rolling operation.
- This ensures each row only uses historical information.


In [ ]:
train_m = train_m.sort_values(["Store", "Date"]).copy()

g = train_m.groupby("Store", group_keys=False)

train_m["Sales_lag_1"] = g["Sales"].shift(1)
train_m["Sales_lag_7"] = g["Sales"].shift(7)
train_m["Sales_roll7_mean"] = g["Sales"].shift(1).rolling(7).mean().reset_index(level=0, drop=True)
train_m["Sales_roll30_mean"] = g["Sales"].shift(1).rolling(30).mean().reset_index(level=0, drop=True)
train_m["Customers_lag_1"] = g["Customers"].shift(1)
train_m["Customers_roll7_mean"] = g["Customers"].shift(1).rolling(7).mean().reset_index(level=0, drop=True)

lag_cols = [
    "Sales_lag_1", "Sales_lag_7", "Sales_roll7_mean", "Sales_roll30_mean",
    "Customers_lag_1", "Customers_roll7_mean"
]

# Conservative fill for early rows where lag history is unavailable.
for c in lag_cols:
    train_m[c] = train_m[c].fillna(0)


## Baseline Cleaning for Modeling


In [ ]:
# Keep only open days with positive sales for supervised training target.
train_feat = train_m[(train_m["Open"] == 1) & (train_m["Sales"] > 0)].copy()

# For test/scoring data, keep all rows and preserve Id.
test_feat = test_m.copy()

# Optional derived target for models using log scale.
train_feat["log_sales"] = np.log1p(train_feat["Sales"])

print("train_feat:", train_feat.shape)
print("test_feat :", test_feat.shape)


## Quick Feature Quality Checks


In [ ]:
# Missingness snapshot for top columns.
miss_train = train_feat.isna().mean().sort_values(ascending=False)
miss_test = test_feat.isna().mean().sort_values(ascending=False)

print("Train missing > 0 (top 15):")
print(miss_train[miss_train > 0].head(15))

print("\nTest missing > 0 (top 15):")
print(miss_test[miss_test > 0].head(15))


In [ ]:
# Visual sanity check for engineered lag signal.
sample_store = train_feat["Store"].mode()[0]
plot_df = train_feat[train_feat["Store"] == sample_store].sort_values("Date").tail(180)

plt.figure(figsize=(12, 4))
plt.plot(plot_df["Date"], plot_df["Sales"], label="Sales", alpha=0.8)
plt.plot(plot_df["Date"], plot_df["Sales_roll7_mean"], label="Sales_roll7_mean", linewidth=2)
plt.title(f"Store {sample_store}: Sales vs Rolling 7-Day Mean")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fe_store_sales_rolling_signal.png", dpi=140)
plt.show()


## Export Feature Tables


In [ ]:
train_out = PROCESSED_DIR / "train_features.csv"
test_out = PROCESSED_DIR / "test_features.csv"

train_feat.to_csv(train_out, index=False)
test_feat.to_csv(test_out, index=False)

print("Saved:", train_out)
print("Saved:", test_out)


## Next Steps
- Use `train_features.csv` for statistical and ML modeling.
- Build temporally-aware validation splits in downstream notebooks.
